In [1]:
import os
from dotenv import load_dotenv
from entsoe import EntsoePandasClient
import pandas as pd

In [2]:
load_dotenv()
entsoe_api_key = os.getenv("ENTSOE_API_KEY")

client = EntsoePandasClient(api_key=entsoe_api_key)

start = pd.Timestamp("20260701", tz="Europe/Warsaw")
end = pd.Timestamp("20260801", tz="Europe/Warsaw")
country_code = "PL"

# ENTSO-E Transparency Platform

The ENTSO-E Transparency Platform is the central electricity market transparency platform operated by the European Network of Transmission System Operators for Electricity (ENTSO-E) in accordance with Commission Regulation (EU) No. 543/2013. It provides standardized electricity-system and wholesale-market data collected from Transmission System Operators (TSOs), power exchanges, and other qualified data providers across Europe.

## Description of Data from the ENTSO-E API

The following table provides an overview of the main variables obtained from the ENTSO-E API. The selection and terminology are based primarily on the official [Manual of Procedures (MoP)](https://www.entsoe.eu/data/transparency-platform/mop/#mop-revisions) and the corresponding data publications of the ENTSO-E Transparency Platform.

The **day-ahead electricity price** is the **target variable** in the model. All other variables are used as model features.

| Category | Data | API Method Name | Unit | Market Time Unit (MTU) |
|---|---|---|---|---|
| **Information relating to the use of cross-zonal capacities** | Day-ahead electricity prices **(dependent variable)** | `query_day_ahead_prices` | EUR/MWh | 15 min / 1 hour |
| | Physical cross-border flow (export) | `query_physical_crossborder_allborders` (`export=True`) | MW | 15 min / 1 hour |
| | Physical cross-border flow (import) | `query_physical_crossborder_allborders` (`export=False`) | MW | 15 min / 1 hour |
| | Scheduled exchanges from explicit and implicit allocations | `query_scheduled_exchanges` | TBA | TBA |
| **Information on total load** | Total load per bidding zone per market time unit | `query_load` | MW | 15 min / 1 hour |
| | Day-ahead forecast of the total load per market time unit | `query_load_forecast` | MW | 15 min / 1 hour |
| **Generation forecast** | Installed generation capacity (aggregated) | `query_installed_generation_capacity` | MW | 1 year |
| **Actual generation** | Aggregated generation per generation type | `query_generation` | MW | 15 min / 1 hour |
| **Information relating to the unavailability of generation and production units** | Planned and actual unavailability of a generation unit | `query_unavailability_of_generation_units` | MW | 1 hour |
| | Planned and actual unavailability of a production unit | `query_unavailability_of_production_units` | MW | 1 hour |

**Note:** The Single Day-Ahead Coupling (SDAC) transitioned from hourly to 15-minute Market Time Units (MTUs) in September 2025. The go-live took place on trading day **30 September 2025**, with the first delivery day under the new 15-minute MTU being **1 October 2025**. For the purposes of this project, all data available at a 15-minute resolution will be aggregated to a **1-hour resolution**. This ensures a consistent resolution across all variables used in the model.

## 1. Information relating to the use of cross zonal capacities

### 1.1 Energy Prices

**Description:**

Day-ahead electricity prices for each bidding zone, expressed in €/MWh at the market time-unit resolution. These prices represent the outcome of the day-ahead market and are the target variable for the forecasting analysis.

**Publication deadline for ENTSO-E:**

It shall be published no later than one hour after gate closure.

In [3]:
day_ahead_prices = client.query_day_ahead_prices(
    country_code=country_code, start=start, end=end
)
pd.DataFrame(day_ahead_prices, columns=["day_ahead_prices"])

,day_ahead_prices
2026-07-01 00:00:00+02:00,151.12
2026-07-01 00:15:00+02:00,154.00
2026-07-01 00:30:00+02:00,145.86
2026-07-01 00:45:00+02:00,137.11
2026-07-01 01:00:00+02:00,146.74
...,...
2026-07-31 23:00:00+02:00,178.63
2026-07-31 23:15:00+02:00,170.00
2026-07-31 23:30:00+02:00,167.36
2026-07-31 23:45:00+02:00,142.61


### 1.2 Physical Flows (cross-border)

**Description:**

Physical flows between bidding zones per market time unit.

**Publication deadline for Entso-E:**

At the latest H+1 after the end of the operating period

In [4]:
physical_crossborder_allborders_export = client.query_physical_crossborder_allborders(
    country_code=country_code, start=start, end=end, export=True
)
physical_crossborder_allborders_export

,CZ,DE_LU,LT,SE_4,SK,UA,sum
2026-07-01 00:00:00+02:00,1315.92,26.529,79.1,0.0,760.8,80.4,2262.749
2026-07-01 00:15:00+02:00,1458.94,8.340,0.0,0.0,793.2,97.5,2357.980
2026-07-01 00:30:00+02:00,1425.53,51.970,161.4,0.0,694.1,78.2,2411.200
2026-07-01 00:45:00+02:00,1332.23,9.767,153.2,0.0,762.7,124.9,2382.797
2026-07-01 01:00:00+02:00,1438.38,99.473,125.4,0.0,805.3,117.7,2586.253
...,...,...,...,...,...,...,...
2026-07-31 22:45:00+02:00,851.90,0.000,0.0,0.0,254.5,0.0,1106.400
2026-07-31 23:00:00+02:00,874.76,0.000,0.0,0.0,305.6,39.4,1219.760
2026-07-31 23:15:00+02:00,813.46,0.000,0.0,0.0,297.6,21.5,1132.560
2026-07-31 23:30:00+02:00,866.25,0.000,0.0,0.0,322.5,28.4,1217.150


In [5]:
physical_crossborder_allborders_import = client.query_physical_crossborder_allborders(
    country_code=country_code, start=start, end=end, export=False
)
physical_crossborder_allborders_import

,CZ,DE_LU,LT,SE_4,SK,UA,sum
2026-07-01 00:00:00+02:00,0.0,344.778,0.0,87.3,0.0,0.0,432.078
2026-07-01 00:15:00+02:00,0.0,711.997,13.3,0.0,0.0,0.0,725.297
2026-07-01 00:30:00+02:00,0.0,736.544,0.0,0.0,0.0,0.0,736.544
2026-07-01 00:45:00+02:00,0.0,870.704,0.0,0.0,0.0,0.0,870.704
2026-07-01 01:00:00+02:00,0.0,663.474,0.0,117.5,0.0,0.0,780.974
...,...,...,...,...,...,...,...
2026-07-31 22:45:00+02:00,0.0,878.707,14.5,487.4,0.0,13.6,1394.207
2026-07-31 23:00:00+02:00,0.0,1142.110,176.8,487.8,0.0,0.0,1806.710
2026-07-31 23:15:00+02:00,0.0,1174.230,195.7,487.9,0.0,0.0,1857.830
2026-07-31 23:30:00+02:00,0.0,878.650,150.4,487.6,0.0,0.0,1516.650


### 1.3 Scheduled exchanges from explicit and implicit allocations

## 2. Information on total load

### 2.1 Total load per bidding zone per market time unit
**Description:** 

Actual total electricity load for each bidding zone, expressed in MW at the market time-unit resolution. This variable represents the realized electricity demand and can be used as an explanatory variable in the forecasting analysis.

**Publication deadline for ENTSO-E:**

Publication based on market time unit. At the latest H+1 after the end of the operating period (of one market time unit length).

In [6]:
load = client.query_load(country_code=country_code, start=start, end=end)
load

,Actual Load
2026-07-01 00:00:00+02:00,17016.174
2026-07-01 00:15:00+02:00,16601.641
2026-07-01 00:30:00+02:00,16472.105
2026-07-01 00:45:00+02:00,16255.833
2026-07-01 01:00:00+02:00,16093.285
...,...
2026-07-31 22:45:00+02:00,17694.037
2026-07-31 23:00:00+02:00,17422.265
2026-07-31 23:15:00+02:00,17218.118
2026-07-31 23:30:00+02:00,16859.868


### 2.2 Day-ahead forecast of the total load per market time unit

**Description:**

Day-ahead forecasts of total electricity load for each bidding zone, expressed in MW at the market time-unit resolution. These forecasts are published before delivery and therefore represent information that can be available at the forecast time.

**Publication deadline for ENTSO-E:**

Publication is necessary in due time for the negotiation of all transactions: D-1, at the latest 2 hours before the gate closure time of the day-ahead market in the bidding area. If the gate closure doesn’t exists in the bidding area then the publication time is D-1, at 12:00 in local time zone.

In [7]:
load_forecast = client.query_load_forecast(
    country_code=country_code, start=start, end=end
)
load_forecast

,Forecasted Load
2026-07-01 00:00:00+02:00,16950.0
2026-07-01 00:15:00+02:00,16700.0
2026-07-01 00:30:00+02:00,16500.0
2026-07-01 00:45:00+02:00,16250.0
2026-07-01 01:00:00+02:00,16100.0
...,...
2026-07-31 22:45:00+02:00,18500.0
2026-07-31 23:00:00+02:00,18150.0
2026-07-31 23:15:00+02:00,17700.0
2026-07-31 23:30:00+02:00,17350.0


## 3. Forecast generation

### 3.1 Installed Generation Capacity aggregated

**Detailed description:**

Installed electricity generation capacity aggregated by generation type for each bidding zone, expressed in MW. This dataset provides information on the available generation infrastructure and its composition across technologies.

**Publication deadline for ENTSO-E**

One week before the first year to which the data refers.

In [8]:
installed_generation_capacity = client.query_installed_generation_capacity(
    country_code=country_code, start=start, end=end
)
installed_generation_capacity

,Biomass,Fossil Brown coal/Lignite,Fossil Coal-derived gas,Fossil Gas,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Hydro Water Reservoir,Other,Solar,Wind Onshore
2026-01-02 00:00:00+01:00,698.294,6951.0,499.254,5430.652,18542.593,405.035,1524.774,322.755,467.895,1979.566,20734.092,10699.706


## 4. Actual generation

### 4.1 Aggregated generation per type

**Detailed description:**

Actual electricity generation aggregated by generation type for each bidding zone, expressed in MW at the market time-unit resolution. The data describe realized generation from different technologies, including renewable and conventional sources.

**Publication deadline for ENTSO-E:**

H+1 following the concerned MTU

In [9]:
generation = client.query_generation(country_code=country_code, start=start, end=end)
generation

Biomass Fossil Brown coal/Lignite  \
                          Actual Aggregated         Actual Aggregated   
2026-07-01 00:00:00+02:00           252.158                  4592.465   
2026-07-01 00:15:00+02:00           255.250                  4593.888   
2026-07-01 00:30:00+02:00           259.395                  4562.371   
2026-07-01 00:45:00+02:00           262.238                  4548.415   
2026-07-01 01:00:00+02:00           264.532                  4568.983   
...                                     ...                       ...   
2026-07-31 22:45:00+02:00           336.107                  4142.211   
2026-07-31 23:00:00+02:00           331.877                  4192.454   
2026-07-31 23:15:00+02:00           327.241                  4223.486   
2026-07-31 23:30:00+02:00           324.023                  4263.752   
2026-07-31 23:45:00+02:00           318.734                  4121.164   

                          Fossil Coal-derived gas        Fossil Gas  \
                                Actual Aggregated Actual Aggregated   
2026-07-01 00:00:00+02:00                 175.683          2654.429   
2026-07-01 00:15:00+02:00                 177.325          2622.002   
2026-07-01 00:30:00+02:00                 175.968          2525.814   
2026-07-01 00:45:00+02:00                 177.824          2347.992   
2026-07-01 01:00:00+02:00                 174.975          2661.141   
...                                           ...               ...   
2026-07-31 22:45:00+02:00                 176.440          2244.025   
2026-07-31 23:00:00+02:00                 175.444          1766.314   
2026-07-31 23:15:00+02:00                 174.187          1752.616   
2026-07-31 23:30:00+02:00                 174.285          1830.260   
2026-07-31 23:45:00+02:00                 174.028          1877.508   

                           Fossil Hard coal        Fossil Oil  \
                          Actual Aggregated Actual Aggregated   
2026-07-01 00:00:00+02:00          8105.549           118.421   
2026-07-01 00:15:00+02:00          7859.902           118.107   
2026-07-01 00:30:00+02:00          7603.986           118.368   
2026-07-01 00:45:00+02:00          7578.719           118.157   
2026-07-01 01:00:00+02:00          7533.557           117.799   
...                                     ...               ...   
2026-07-31 22:45:00+02:00          7652.209           139.506   
2026-07-31 23:00:00+02:00          7426.906           138.469   
2026-07-31 23:15:00+02:00          7230.950           139.814   
2026-07-31 23:30:00+02:00          7207.841           140.492   
2026-07-31 23:45:00+02:00          6971.608           139.458   

                          Hydro Pumped Storage                     \
                             Actual Aggregated Actual Consumption   
2026-07-01 00:00:00+02:00                  0.0             15.533   
2026-07-01 00:15:00+02:00                  0.0             15.915   
2026-07-01 00:30:00+02:00                  0.0             15.839   
2026-07-01 00:45:00+02:00                  0.0             16.106   
2026-07-01 01:00:00+02:00                  0.0             15.725   
...                                        ...                ...   
2026-07-31 22:45:00+02:00                375.9              6.077   
2026-07-31 23:00:00+02:00                316.6             12.536   
2026-07-31 23:15:00+02:00                314.7             12.612   
2026-07-31 23:30:00+02:00                152.0             12.306   
2026-07-31 23:45:00+02:00                133.7             17.658   

                          Hydro Run-of-river and poundage  \
                                        Actual Aggregated   
2026-07-01 00:00:00+02:00                          52.339   
2026-07-01 00:15:00+02:00                          52.239   
2026-07-01 00:30:00+02:00                          52.239   
2026-07-01 00:45:00+02:00                          51.541   
2026-07-01 01:00:00+02:00                          5

## 5. Information relating to the unavailability of generation and production units

### 5.1 Planned and Current Unavailability of Generation Units

**Description:**

Planned unavailability of 100 MW or more of a generation unit, including changes of 100 MW or more in the planned unavailability of that generation unit, expected to last for at least one market time unit and up to three years ahead. The dataset also covers changes of 100 MW or more in the actual availability of a generation unit, expected to last for at least one market time unit.

**Publication Deadline for ENTSO-E:**

For planned unavailability, the information shall be published H+1 at the latest after the plan is approved. For changes in actual availability, the information shall be published no later than H+1 after the change in actual availability.

In [10]:
unavailability_of_generation_units = client.query_unavailability_of_generation_units(
    country_code="PL", start=start, end=end
)
unavailability_of_generation_units

,avail_qty,biddingzone_domain,businesstype,curvetype,docstatus,end,mrid,nominal_power,plant_type,production_resource_id,production_resource_location,production_resource_name,production_resource_psr_name,pstn,qty_uom,resolution,revision,start
created_doc_time,,,,,,,,,,,,,,,,,,
2025-10-05 20:08:53+02:00,0,PL,Planned maintenance,A03,None,2026-07-15 00:00:00+02:00,vzLRWDmON0yYWXg8VlpGxA,107.0,Fossil Hard coal,19W0000000000806,Polska,Kraków Łęg,Kraków Łęg B1,1,MAW,PT1M,1,2026-06-15 00:01:00+02:00
2025-10-05 21:14:00+02:00,0,PL,Planned maintenance,A03,None,2026-07-31 00:00:00+02:00,yZnvwsnYMb48OjYw9qaD1Q,112.0,Fossil Hard coal,19W000000000076Y,Polska,Katowice,Katowice B1,1,MAW,PT1M,1,2026-07-01 00:01:00+02:00
2025-10-06 01:11:13+02:00,0,PL,Planned maintenance,A03,None,2026-07-07 16:00:00+02:00,zSDsz4TEUEDgPmgSVQFDhg,104.0,Fossil Hard coal,19W0000000000369,Polska,Chorzów,Chorzów B2,1,MAW,PT1M,1,2026-06-06 02:01:00+02:00
2025-10-06 13:38:03+02:00,0,PL,Planned maintenance,A03,Cancelled,2026-08-20 00:00:00+02:00,mQmes-XWN9f-UhdUepktZQ,226.0,Fossil Hard coal,19W0000000001519,Polska,Połaniec,Połaniec B4,1,MAW,PT1M,4,2026-07-05 00:01:00+02:00
2025-10-07 01:28:14+02:00,0,PL,Planned maintenance,A03,None,2026-08-28 00:00:00+02:00,TCJod7mQ5DATJrP0yAUsOg,105.9,Fossil Hard coal,19W0000000002191,Polska,EC Siekierki,Siekierki B08,1,MAW,PT1M,2,2026-03-02 00:01:00+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-31 17:13:19+02:00,305.1,PL,Unplanned outage,A03,None,2026-07-31 23:00:00+02:00,gGlRbGYuhYx2SWZcY5FBUw,430.1,Fossil Hard coal,19W000000000111L,Polska,Łagisza,Łagisza B10,1,MAW,PT1M,1,2026-07-31 16:54:00+02:00
2026-08-04 09:11:43+02:00,0,PL,Unplanned outage,A03,None,2026-08-04 09:30:00+02:00,5jbVZKdtQN9ZXx-eHLqCXw,226.0,Fossil Hard coal,19W0000000001519,Polska,Połaniec,Połaniec B3,1,MAW,PT1M,4,2026-07-29 15:31:00+02:00
2026-08-04 11:05:26+02:00,0,PL,Unplanned outage,A03,None,2026-08-04 11:00:00+02:00,HJSkYNyeIYSlNmrC3ciR3A,210.0,Fossil Hard coal,19W000000000095U,Polska,Kozienice 1,Kozienice 1 B2,1,MAW,PT1M,3,2026-07-29 12:00:00+02:00


### 5.2 Planned and Current Unavailability of Production Units

**Description:**

Planned unavailability of a production unit with an installed generation capacity of 200 MW or more, including changes of 100 MW or more in the planned unavailability of that production unit, but not already published as a generation-unit unavailability. The event is expected to last for at least one market time unit and up to three years ahead. The dataset also covers changes of 100 MW or more in the actual availability of a production unit with an installed generation capacity of 200 MW or more.

**Publication Deadline for ENTSO-E:**

For planned unavailability, the information shall be published H+1 at the latest after the plan is approved. For changes in actual availability, the information shall be published no later than H+1 after the change in actual availability.

In [11]:
unavailability_of_production_units = client.query_unavailability_of_production_units(
    country_code="PL", start=start, end=end
)
unavailability_of_production_units

,avail_qty,biddingzone_domain,businesstype,curvetype,docstatus,end,mrid,nominal_power,plant_type,production_resource_id,production_resource_location,production_resource_name,production_resource_psr_name,pstn,qty_uom,resolution,revision,start
created_doc_time,,,,,,,,,,,,,,,,,,
2025-10-07 01:28:14+02:00,493,PL,Planned maintenance,A03,Cancelled,2026-07-18 00:00:00+02:00,6BeWZLwdz6DKKSVP0r0waA,533.8,Fossil Hard coal,19W0000000002191,Polska,EC Siekierki,,1,MAW,PT1M,2,2026-06-22 00:01:00+02:00
2025-10-07 01:28:18+02:00,441,PL,Planned maintenance,A03,Cancelled,2026-08-15 00:00:00+02:00,Bazhc8LjP9Z-ZOWDLRAy1g,533.8,Fossil Hard coal,19W0000000002191,Polska,EC Siekierki,,1,MAW,PT1M,3,2026-06-12 00:01:00+02:00
2025-10-07 01:28:18+02:00,473,PL,Planned maintenance,A03,Cancelled,2026-08-15 00:00:00+02:00,A4bzEQ9CvIvYcgtxTRan-Q,533.8,Fossil Hard coal,19W0000000002191,Polska,EC Siekierki,,1,MAW,PT1M,2,2026-07-18 00:01:00+02:00
2025-10-07 01:28:19+02:00,473,PL,Planned maintenance,A03,Cancelled,2026-08-18 00:00:00+02:00,dNyXiN4oHkpre1YvVzuwUA,533.8,Fossil Hard coal,19W0000000002191,Polska,EC Siekierki,,1,MAW,PT1M,3,2026-06-12 00:01:00+02:00
2025-10-07 01:28:19+02:00,505,PL,Planned maintenance,A03,None,2026-08-18 00:00:00+02:00,5az3Te-5EIACvlWo89amXg,533.8,Fossil Hard coal,19W0000000002191,Polska,EC Siekierki,,1,MAW,PT1M,1,2026-06-29 00:01:00+02:00
2025-10-07 01:28:26+02:00,167,PL,Planned maintenance,A03,Cancelled,2026-07-14 00:00:00+02:00,zuQqZvCjJL-LywXDaG11cQ,245.8,Fossil Hard coal,19W000000000297I,Polska,EC Żerań 1,,1,MAW,PT1M,2,2026-07-01 00:01:00+02:00
2025-10-07 01:28:26+02:00,132,PL,Planned maintenance,A03,Cancelled,2026-07-14 00:00:00+02:00,5ipHuSOxDnPnSwCXEPvNEg,245.8,Fossil Hard coal,19W000000000297I,Polska,EC Żerań 1,,1,MAW,PT1M,2,2026-07-01 00:01:00+02:00
2025-10-07 01:28:26+02:00,194,PL,Planned maintenance,A03,None,2026-07-14 00:00:00+02:00,xPpwLkHyVoCZ9mjgt206yw,245.8,Fossil Hard coal,19W000000000297I,Polska,EC Żerań 1,,1,MAW,PT1M,1,2026-07-01 00:01:00+02:00
2025-10-07 01:28:26+02:00,97,PL,Planned maintenance,A03,Cancelled,2026-07-14 00:00:00+02:00,5pSyD0R2sWFsMHlUd3GnDw,245.8,Fossil Hard coal,19W000000000297I,Polska,EC Żerań 1,,1,MAW,PT1M,2,2026-07-01 00:01:00+02:00
